# Kenya Gazette PDF to structured JSON (Docling + Spatial Reorder)

This notebook runs **Docling** on one or more Gazette PDFs and builds a **single JSON-friendly record per file** with:

- PDF metadata: inferred title, file name, size in bytes, page count
- Full **Docling** exports: plain text, Markdown, and a compact document summary (full `export_to_dict` is optional — it can be large)
- **Spatial reading-order correction**: reorders text elements using bounding-box coordinates to fix two-column reading order (left column top-to-bottom, then right column, then full-width zones)
- **Gazette notices** split by the phrase `GAZETTE NOTICE NO.` (Kenya-style), with notice number, header line, and full notice text

**Input:** drop PDFs into the `pdfs/` folder next to this notebook (or change `PDF_DIR` below). Use **Choose which PDFs** to run on every file or only a chosen subset.

> **See also:** `gazette_docling_pipeline.ipynb` for the original pipeline without spatial reordering.

## Environment

Use the project virtual environment (already created in this folder as `.venv`):

- **Windows (PowerShell):** `.\.venv\Scripts\Activate.ps1`
- **Then:** `python -m pip install -r requirements.txt` if needed
- **Jupyter kernel:** register with `python -m ipykernel install --user --name=gazette-docling --display-name="Python (gazette-docling)"` using the venv’s `python`.

Select that kernel in this notebook before running cells.

In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from docling.document_converter import DocumentConverter
from docling_core.types.doc.labels import DocItemLabel

from kenya_gazette_parser import __version__ as LIBRARY_VERSION
from kenya_gazette_parser.identity import (
    SCHEMA_VERSION,
    make_extracted_at,
    compute_pdf_sha256,
    make_gazette_issue_id,
    make_notice_id,
)
from kenya_gazette_parser.masthead import parse_masthead
from kenya_gazette_parser.spatial import (
    reorder_by_spatial_position,
    reorder_by_spatial_position_with_confidence,
    compute_page_layout_confidence,
)
from kenya_gazette_parser.splitting import split_gazette_notices
from kenya_gazette_parser.trailing import detect_trailing_content_cutoff
from kenya_gazette_parser.corrigenda import extract_corrigenda
from kenya_gazette_parser.scoring import (
    score_notice_number, score_structure, score_spatial, score_boundary,
    score_table, composite_confidence, score_notice, score_notices,
    compute_document_confidence, aggregate_confidence, filter_notices,
    partition_by_band, explain,
)
from kenya_gazette_parser.envelope_builder import build_envelope_dict
from kenya_gazette_parser.pipeline import build_envelope
from kenya_gazette_parser.models import Envelope

PROJECT_ROOT = Path.cwd().resolve()
PDF_DIR = PROJECT_ROOT / "pdfs"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF_DIR:", PDF_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("LIBRARY_VERSION:", LIBRARY_VERSION, "SCHEMA_VERSION:", SCHEMA_VERSION)


In [ ]:
# F11 masthead parsing has moved to kenya_gazette_parser.masthead.parse_masthead.
# The old inline smoke test (test_parse_masthead) is retained by the canonical-PDF
# regression in scripts/f20_regression_check.py; there is no notebook-local
# duplicate to keep in sync.


## Gazette notice splitting

Notices are detected by a **strict full-line, all-caps** match on `GAZETTE NOTICE NO. <digits>` (with tolerance for the OCR typo `GAZETE`). This avoids false positives from inline references like "IN Gazette Notice No. 14152 …" that appear inside prose.

Each notice is then structured with:
- **`title_lines`** — heading lines before the statutory body (e.g. act name, chapter reference, subtitle)
- **`body_segments`** — typed blocks (`text`, `table`, `blank`) detected via multi-space column heuristics
- **`derived_table`** — optional structured rows when an S/No–Name–Position table is detected
- **Running-header stripping** — page furniture (page numbers, running headers, dates) is removed from notice bodies

**Corrigenda** (correction notices) are extracted separately from the preamble section before the first notice. Each corrigendum captures the referenced notice number/year and the error/correction text.

You can tune `NOTICE_HEAD_RE` if your issues use a different wording.

In [ ]:
# Notice splitting, corrigenda extraction, and trailing-content detection have
# moved to kenya_gazette_parser.{splitting,corrigenda,trailing}. The helpers
# below (`extract_title_from_docling`, `docling_export_summary`,
# `highlight_gazette_notices_in_markdown`) are notebook-only demo utilities;
# they stay here until F21 introduces write_envelope.


def extract_title_from_docling(doc) -> str:
    for item in getattr(doc, "texts", []) or []:
        if getattr(item, "label", None) == DocItemLabel.TITLE and getattr(item, "text", None):
            return str(item.text).strip()
    return ""


def docling_export_summary(doc_dict: dict[str, Any]) -> dict[str, Any]:
    """Small fingerprint of the Docling JSON without dumping huge trees twice."""
    texts = doc_dict.get("texts") or []
    return {
        "schema_name": doc_dict.get("schema_name"),
        "version": doc_dict.get("version"),
        "name": doc_dict.get("name"),
        "texts_count": len(texts) if isinstance(texts, list) else None,
        "tables_count": len(doc_dict.get("tables") or []) if isinstance(doc_dict.get("tables"), list) else None,
        "pictures_count": len(doc_dict.get("pictures") or []) if isinstance(doc_dict.get("pictures"), list) else None,
        "pages_count": len(doc_dict.get("pages") or []) if isinstance(doc_dict.get("pages"), list) else None,
    }


_GAZETTE_NOTICE_MD_LINE = re.compile(
    r"^(\#\# )?(GAZETTE NOTICE NO\. \d+)\s*$",
    re.MULTILINE,
)

_GAZETTE_NOTICE_HIGHLIGHT_STYLE = (
    'style="background-color:#fff3cd;color:#1a1a1a;padding:0.15em 0.35em;'
    'border-radius:3px;font-weight:600;"'
)


def highlight_gazette_notices_in_markdown(md: str) -> str:
    """Wrap standalone GAZETTE NOTICE NO. lines for visible highlight in Markdown HTML preview."""

    def repl(m: re.Match) -> str:
        notice = m.group(2)
        inner = f'<span {_GAZETTE_NOTICE_HIGHLIGHT_STYLE}>{notice}</span>'
        if m.group(1):
            return f"## {inner}"
        return inner

    return _GAZETTE_NOTICE_MD_LINE.sub(repl, md)


## Confidence scoring

Rule-based confidence for every extracted notice along five dimensions:

- **`notice_number`** -- is the captured number a plausible digit string?
- **`structure`** -- does the body look complete (length, legal markers, signature)?
- **`spatial`** -- does the text flow read cleanly (no mid-sentence breaks, no repeated chunks)?
- **`boundary`** -- was the header a strict match, does the span end with punctuation?
- **`table`** -- for notices that produced a `derived_table`, are the rows well-formed?

Every scorer returns `(score, reasons)`. The reasons list is what makes low scores auditable.

The `composite` is a weighted mean (see `docs/data-quality-confidence-scoring.md`). `table` is only folded in when a `derived_table` is present.

Two thresholds drive downstream use:

- `composite >= 0.80` -- high confidence, usable without review.
- `0.50 <= composite < 0.80` -- medium, spot-check.
- `composite < 0.50` -- likely broken, needs review (and is the gate for optional LLM semantic validation).

In [ ]:
# Confidence scoring has moved to kenya_gazette_parser.scoring:
#   score_notice_number, score_structure, score_spatial, score_boundary,
#   score_table, composite_confidence, score_notice, score_notices,
#   compute_document_confidence. See the top-of-notebook imports.


In [ ]:
# Downstream confidence helpers (filter_notices, partition_by_band,
# aggregate_confidence, explain) have moved to kenya_gazette_parser.scoring.


## Spatial reading-order correction

Docling's internal reading-order predictor uses heuristics that can misorder text in two-column layouts — a notice that starts at the bottom of the left column and continues at the top of the right column may be split or interleaved with unrelated content.

The `reorder_by_spatial_position()` function fixes this by using each element's bounding-box coordinates from the Docling export dict:

1. **Group** all text (and table) elements by page number
2. **Detect columns** by checking whether element centers fall left or right of the page midpoint
3. **Find the column-zone boundary** — the y-coordinate where the shorter column ends
4. **Reclassify** elements below that boundary as full-width (even if left-aligned)
5. **Sort**: furniture (headers) → left column top-to-bottom → right column top-to-bottom → full-width zone top-to-bottom
6. **Concatenate** to produce corrected plain text

In [ ]:
# Spatial reading-order reorder + layout confidence have moved to
# kenya_gazette_parser.spatial. D2 fix: `reorder_by_spatial_position_with_confidence`
# no longer emits `n_pages`; `envelope_builder.build_envelope_dict` now
# passes `layout_info` through verbatim.


## Convert pipeline (with spatial reorder)

`DocumentConverter` is created once and reused. For each PDF we populate the output record and optionally write the full Docling JSON to `output/<stem>_docling.json`.

Gazette notices are now split from the **spatially reordered** plain text instead of the raw `export_to_text()` output. Both versions are kept in the record for comparison.

In [ ]:
# Identity and versioning helpers have moved to:
#   * kenya_gazette_parser.__version__ (LIBRARY_VERSION, SCHEMA_VERSION)
#   * kenya_gazette_parser.identity (make_extracted_at, compute_pdf_sha256,
#     make_gazette_issue_id, make_notice_id).
# D1 fix: the notebook no longer declares LIBRARY_VERSION / SCHEMA_VERSION;
# both are imported from the package at the top of this notebook.


In [ ]:
# F20: `process_pdf` is now a thin shim over `kenya_gazette_parser.pipeline.build_envelope`.
# The pure-compute path lives in the library; the notebook demo still writes the
# same five output/{stem}/ side files as before (F21 will move those into
# `write_envelope`). `envelope_builder.build_envelope_dict` and
# `_estimate_ocr_quality` have also moved into the package.


@dataclass
class GazettePipeline:
    converter: DocumentConverter = field(default_factory=DocumentConverter)
    include_full_docling_dict: bool = False

    def process_pdf(self, pdf_path: Path) -> dict[str, Any]:
        pdf_path = Path(pdf_path).resolve()

        env = build_envelope(
            pdf_path,
            converter=self.converter,
            include_full_docling_dict=self.include_full_docling_dict,
        )
        record = env.model_dump(mode="json")

        pdf_output_dir = OUTPUT_DIR / pdf_path.stem
        pdf_output_dir.mkdir(parents=True, exist_ok=True)

        out_json = pdf_output_dir / f"{pdf_path.stem}_gazette_spatial.json"
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        # Regenerate the diagnostic side files from a second pass through Docling.
        # This keeps the notebook demo byte-for-byte compatible with the pre-F20
        # output tree (Docling conversion is deterministic for these files).
        result = self.converter.convert(str(pdf_path))
        doc = result.document
        plain = doc.export_to_text()
        md = doc.export_to_markdown()
        doc_dict = doc.export_to_dict()
        plain_spatial, _layout = reorder_by_spatial_position_with_confidence(doc_dict)

        out_spatial_txt = pdf_output_dir / f"{pdf_path.stem}_spatial.txt"
        with open(out_spatial_txt, "w", encoding="utf-8") as f:
            f.write(plain_spatial)

        out_markdown_default = pdf_output_dir / f"{pdf_path.stem}_docling_markdown.md"
        with open(out_markdown_default, "w", encoding="utf-8") as f:
            f.write(highlight_gazette_notices_in_markdown(md))

        out_markdown_spatial = pdf_output_dir / f"{pdf_path.stem}_spatial_markdown.md"
        with open(out_markdown_spatial, "w", encoding="utf-8") as f:
            f.write(highlight_gazette_notices_in_markdown(plain_spatial))

        docling_only = pdf_output_dir / f"{pdf_path.stem}_docling.json"
        with open(docling_only, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

        print(f"Wrote: {out_json}")
        print(f"Wrote: {out_spatial_txt}")
        print(f"Wrote: {out_markdown_default}")
        print(f"Wrote: {out_markdown_spatial}")
        print(f"Wrote: {docling_only}")
        return record


def run_pdfs(pipeline: GazettePipeline, pdf_paths: list[Path]) -> list[dict[str, Any]]:
    """Run the pipeline on an explicit ordered list of PDF paths."""
    if not pdf_paths:
        print("No PDF paths to process.")
        return []
    results: list[dict[str, Any]] = []
    for p in pdf_paths:
        print("\n--- Processing:", p.name, "---")
        results.append(pipeline.process_pdf(p))
    return results


def run_folder(pipeline: GazettePipeline, folder: Path | None = None) -> list[dict[str, Any]]:
    """Process every *.pdf in folder (same as selection mode 'all')."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    return run_pdfs(pipeline, pdfs)


def resolve_pdf_selection(
    mode: str,
    selected_names: list[str],
    folder: Path | None = None,
) -> list[Path]:
    """Resolve 'all' or 'selected' to a list of existing PDF paths under folder."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    m = (mode or "all").strip().lower()
    if m == "all":
        return pdfs
    if m == "selected":
        if not selected_names:
            print("PDF_SELECTION_MODE is 'selected' but SELECTED_PDF_NAMES is empty. Nothing to do.")
            return []
        by_name = {p.name: p for p in pdfs}
        out: list[Path] = []
        missing: list[str] = []
        for raw in selected_names:
            name = raw.strip()
            if not name:
                continue
            p = by_name.get(name)
            if p is None:
                p = by_name.get(Path(name).name)
            if p is None:
                missing.append(name)
            elif p not in out:
                out.append(p)
        if missing:
            print("Warning: not found in", folder, ":", missing)
            print("Available PDFs:", [p.name for p in pdfs])
        return out
    raise ValueError('PDF_SELECTION_MODE must be "all" or "selected".')


## Choose which PDFs to process

Set **`PDF_SELECTION_MODE`** in the next cell:

- **`"all"`** — every `*.pdf` under `PDF_DIR`
- **`"selected"`** — only files listed in **`SELECTED_PDF_NAMES`** (exact file names as they appear in `pdfs/`)

Then run the pipeline cell.

## Run

First conversion may **download models** and take longer. Subsequent runs are faster.

In [10]:
# --- Configure which PDFs to process ---
# When mode is "selected", list exact file names as they appear in PDF_DIR (order is preserved).
# When mode is "all", SELECTED_PDF_NAMES is ignored.
PDF_SELECTION_MODE = "selected"  # "all" | "selected"
SELECTED_PDF_NAMES: list[str] = [
    # "Kenya Gazette Vol CXINo 108.pdf",
    # "Kenya Gazette Vol CXINo 107.pdf",
    # "Kenya Gazette Vol CXINo 106.pdf",
    # "Kenya Gazette Vol CXINo 105.pdf",
    # "Kenya Gazette Vol CXINo 104.pdf",
    # "Kenya Gazette Vol CXINo 103.pdf",
    # "Kenya Gazette Vol CXINo 102.pdf",
    # "Kenya Gazette Vol CXINo 101.pdf",
    "Kenya Gazette Vol CXINo 100.pdf",
    # "Kenya Gazette Vol CXIINo 136.pdf",
    # "Kenya Gazette Vol CXIINo 132.pdf",
    # "Kenya Gazette Vol CXIINo 76.pdf",
    # "Kenya Gazette Vol CVIINo 90.pdf",
    "Kenya Gazette Vol CIINo 83 - pre 2010.pdf",
    # "Kenya Gazette Vol CXXVIIINo 41.pdf",
    # "Kenya Gazette Vol CXXVIIINo 51.pdf",
    # "Kenya Gazette Vol CXXVIINo 255.pdf",
    # "Kenya Gazette Vol CXXVIINo 254.pdf",
    # "Kenya Gazette Vol CXXVIINo 258.pdf",
    # "Kenya Gazette Vol CXXVIINo 107.pdf",
    # "Kenya Gazette Vol CXXVIINo 108.pdf",
    # "Kenya Gazette Vol CXXVIINo 111.pdf",
    "Kenya Gazette Vol CXXVIINo 63.pdf",
    # "Kenya Gazette Vol CXXVIINo 72.pdf",
    # "Kenya Gazette Vol CXXVIINo 73.pdf",
    # "Kenya Gazette Vol CXXVIINo 77.pdf",
    # "Kenya Gazette Vol CXXVIINo 266.pdf",
    "Kenya Gazette Vol CXXIVNo 282.pdf"
]

pipeline = GazettePipeline(include_full_docling_dict=False)
pdf_paths = resolve_pdf_selection(PDF_SELECTION_MODE, SELECTED_PDF_NAMES)
print(
    "Mode:",
    PDF_SELECTION_MODE,
    "|",
    len(pdf_paths),
    "file(s):",
    [p.name for p in pdf_paths],
)
results = run_pdfs(pipeline, pdf_paths)

if results:
    sample = results[0]
    notices = sample.get("gazette_notices") or []
    corrigenda = sample.get("corrigenda") or []
    with_tables = [n for n in notices if "derived_table" in n]
    print("Keys:", list(sample.keys()))
    print("Pages:", sample.get("pages"))
    print(f"Corrigenda count: {len(corrigenda)}")
    print(f"Notice count: {len(notices)}")
    print(f"Notices with derived tables: {len(with_tables)}")
    if corrigenda:
        first_corr = corrigenda[0]
        print(f"\nFirst corrigendum preview:")
        print(f"  Referenced: Notice {first_corr.get('referenced_notice_no')} of {first_corr.get('referenced_year')}")
        print(f"  Error: {first_corr.get('error_text')}")
        print(f"  Correction: {first_corr.get('correction_text')}")
    if notices:
        first = notices[0]
        print(f"\nFirst notice preview:")
        print(f"  No: {first.get('gazette_notice_no')}")
        print(f"  Header: {first.get('gazette_notice_header')}")
        print(f"  Title lines: {first.get('title_lines', [])}")
        segs = first.get("body_segments", [])
        seg_types = [s["type"] for s in segs]
        print(f"  Body segments: {len(segs)} ({', '.join(f'{t}={seg_types.count(t)}' for t in dict.fromkeys(seg_types))})")
    display_snippet = json.dumps(sample, ensure_ascii=False, indent=2)
    if len(display_snippet) > 8000:
        print(display_snippet[:8000] + "\n... [truncated for display; see output JSON files] ...")
    else:
        print(display_snippet)

[INFO] 2026-04-19 22:49:18,488 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-19 22:49:18,489 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-19 22:49:18,519 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-19 22:49:18,519 [RapidOCR] main.py:50: Using C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth


Mode: selected | 1 file(s): ['Kenya Gazette Vol CXXVIINo 63.pdf']

--- Processing: Kenya Gazette Vol CXXVIINo 63.pdf ---


[INFO] 2026-04-19 22:49:18,692 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-19 22:49:18,693 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-19 22:49:18,696 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-19 22:49:18,696 [RapidOCR] main.py:50: Using C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-19 22:49:18,770 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-19 22:49:18,770 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-19 22:49:18,829 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-04-19 22:49:18,829 [RapidOCR] main.py:50: Using C:\

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

RapidOCR returned empty result!
Stage preprocess failed for run 1, pages [68]: std::bad_alloc
Stage preprocess failed for run 1, pages [69]: std::bad_alloc
Stage preprocess failed for run 1, pages [70]: std::bad_alloc
Stage preprocess failed for run 1, pages [71]: std::bad_alloc
Stage preprocess failed for run 1, pages [72]: std::bad_alloc
Stage preprocess failed for run 1, pages [73]: std::bad_alloc
Stage preprocess failed for run 1, pages [74]: std::bad_alloc
Stage preprocess failed for run 1, pages [75]: std::bad_alloc
Stage preprocess failed for run 1, pages [76]: std::bad_alloc
Stage preprocess failed for run 1, pages [77]: std::bad_alloc
Stage preprocess failed for run 1, pages [78]: std::bad_alloc
Stage preprocess failed for run 1, pages [79]: std::bad_alloc
Stage preprocess failed for run 1, pages [80]: std::bad_alloc
Stage preprocess failed for run 1, pages [81]: std::bad_alloc
Stage preprocess failed for run 1, pages [82]: std::bad_alloc
Stage preprocess failed for run 1, pag

Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_gazette_spatial.json
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_spatial.txt
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_docling_markdown.md
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_spatial_markdown.md
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_docling.json
Keys: ['pdf_title', 'pdf_file_name', 'pdf_path', 'pdf_size_bytes', 'pdf_sha256', 'gazette_issue_id', 'library_version', 'schema_version', 'extracted_at', 'warnings', 'pages', 'volume', 'issue_no', 'publication_date', 'supplement_no', 'masthead_text', 'parse_confidence', 'do

## Optional: embed full Docling tree in the main record

Set `include_full_docling_dict=True` if you need the complete `DoclingDocument` as JSON inside `gazette.json` (large).

In [ ]:
# Example: toggle full dict for a single file
# p = PDF_DIR / "your_issue.pdf"
# if p.exists():
#     full_pipeline = GazettePipeline(include_full_docling_dict=True)
#     full_pipeline.process_pdf(p)

## Confidence report

Walks the `output/` tree and produces a triage view:

- Per-PDF table: notice count, counts by band (high / medium / low), mean and min composite.
- Top-N lowest-confidence notices across the corpus with their reasons.
- Optional CSV export to `output/_confidence_report.csv`.

Run after processing one or more PDFs. If a JSON predates the scoring integration it simply shows as unscored.

In [ ]:
import csv


def _iter_output_gazette_jsons(output_root: Path = OUTPUT_DIR):
    """Yield (pdf_stem, json_path, record) for every gazette_spatial.json in output/."""
    for path in sorted(output_root.glob("*/*_gazette_spatial.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                record = json.load(f)
        except (json.JSONDecodeError, OSError) as exc:
            print(f"skip {path.name}: {exc}")
            continue
        yield path.parent.name, path, record


def confidence_report(
    output_root: Path = OUTPUT_DIR,
    top_n_low: int = 20,
    csv_path: Path | None = None,
) -> dict[str, Any]:
    """Summarize confidence across every processed gazette JSON under output_root.

    Prints a per-PDF table and the lowest-confidence notices across the corpus.
    Returns a dict with the numeric summary for programmatic use.
    """
    per_pdf: list[dict[str, Any]] = []
    all_notices: list[tuple[str, dict[str, Any]]] = []

    for stem, path, record in _iter_output_gazette_jsons(output_root):
        notices = record.get("gazette_notices") or []
        scored = [n for n in notices if (n.get("confidence_scores") or {}).get("composite") is not None]
        bands = partition_by_band(notices)
        agg = aggregate_confidence(notices)
        doc_conf = record.get("document_confidence") or {}
        per_pdf.append({
            "pdf": stem,
            "n_notices": len(notices),
            "scored": len(scored),
            "high": len(bands["high"]),
            "medium": len(bands["medium"]),
            "low": len(bands["low"]),
            "mean_composite": agg["mean"],
            "min_composite": agg["min"],
            "layout": doc_conf.get("layout"),
            "ocr_quality": doc_conf.get("ocr_quality"),
            "document_composite": doc_conf.get("composite"),
        })
        for n in scored:
            all_notices.append((stem, n))

    print(
        f"{'PDF':<55} {'N':>4} {'High':>5} {'Med':>4} {'Low':>4} "
        f"{'Mean':>6} {'Min':>6} {'Layout':>7} {'OCR':>5}"
    )
    print("-" * 100)
    for row in per_pdf:
        print(
            f"{row['pdf'][:55]:<55} {row['n_notices']:>4} "
            f"{row['high']:>5} {row['medium']:>4} {row['low']:>4} "
            f"{row['mean_composite']:>6.3f} {row['min_composite']:>6.3f} "
            f"{(row['layout'] or 0):>7.3f} {(row['ocr_quality'] or 0):>5.3f}"
        )

    all_notices.sort(key=lambda t: (t[1].get("confidence_scores") or {}).get("composite", 1.0))
    print(f"\nLowest {min(top_n_low, len(all_notices))} notices across corpus:")
    print("-" * 100)
    for stem, n in all_notices[:top_n_low]:
        c = (n.get("confidence_scores") or {}).get("composite")
        no = n.get("gazette_notice_no") or "?"
        hdr = (n.get("gazette_notice_header") or "")[:60]
        reasons = n.get("confidence_reasons") or []
        print(f"[{c:.3f}] {stem[:40]:<40} #{no:<8} {hdr}")
        for r in reasons[:3]:
            print(f"         {r}")

    if csv_path is not None:
        csv_path = Path(csv_path)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow([
                "pdf", "notice_no", "composite", "notice_number", "structure",
                "spatial", "boundary", "table", "header_match", "reasons",
            ])
            for stem, n in all_notices:
                s = n.get("confidence_scores") or {}
                prov = n.get("provenance") or {}
                w.writerow([
                    stem,
                    n.get("gazette_notice_no"),
                    s.get("composite"),
                    s.get("notice_number"),
                    s.get("structure"),
                    s.get("spatial"),
                    s.get("boundary"),
                    s.get("table"),
                    prov.get("header_match"),
                    " | ".join(n.get("confidence_reasons") or []),
                ])
        print(f"\nWrote CSV: {csv_path}")

    return {"per_pdf": per_pdf, "n_notices_total": len(all_notices)}


# Uncomment to run a report over every processed PDF:
_rep = confidence_report(csv_path=OUTPUT_DIR / "_confidence_report.csv")

## Optional: LLM semantic validation

For notices whose rule-based composite is below a threshold (default 0.70) a lightweight LLM can check for scrambled text, merged notices, truncation, and legal incoherence -- things regex cannot see.

- Set `ENABLE_LLM_VALIDATION = True` below.
- Requires `OPENAI_API_KEY` (for OpenAI) or `ANTHROPIC_API_KEY` (for Claude) in the environment.
- Results are cached by body hash under `.llm_cache/` so reruns are free.
- Writes `llm_validation` on each scored notice, plus `composite_enhanced` and `needs_human_review`.

Call `enhance_with_llm(notices)` on any already-scored notice list to update it in place.

In [ ]:
# %pip install openai

In [11]:
import hashlib
import os

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

ENABLE_LLM_VALIDATION = True
LLM_CONFIDENCE_THRESHOLD = 0.80
LLM_MODEL = "gpt-4o-mini"
LLM_CACHE_DIR = PROJECT_ROOT / ".llm_cache"

_LLM_PROMPT = """You are validating an extracted Kenya Gazette notice.
Check for these issues:
1. Text coherence -- does the text flow naturally or appear scrambled from column-mixing?
2. Completeness -- does the notice end with a date and signature, or is it truncated?
3. Single notice -- is this one notice, or were multiple notices merged together?
4. Legal structure -- does it follow gazette patterns (preamble, body, closing)?

Notice:
---
{body}
---

Respond with JSON only:
{{
  "coherence_score": <0.0-1.0>,
  "completeness_score": <0.0-1.0>,
  "single_notice_score": <0.0-1.0>,
  "legal_structure_score": <0.0-1.0>,
  "issues": [<short strings>],
  "needs_human_review": <true/false>
}}
"""


def _llm_cache_path(body: str, model: str) -> Path:
    LLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha1(f"{model}|{body[:4000]}".encode("utf-8")).hexdigest()
    return LLM_CACHE_DIR / f"{key}.json"


def _llm_call_openai(body: str, model: str) -> dict[str, Any]:
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise RuntimeError("pip install openai") from exc
    client = OpenAI()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": _LLM_PROMPT.format(body=body[:4000])}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=300,
    )
    return json.loads(resp.choices[0].message.content or "{}")


def llm_validate_notice(notice: dict[str, Any], model: str = LLM_MODEL) -> dict[str, Any]:
    """Run the LLM validator on a single notice, using the on-disk cache."""
    body = notice.get("gazette_notice_full_text") or ""
    if not body.strip():
        return {"error": "empty body", "needs_human_review": True}

    cache_path = _llm_cache_path(body, model)
    if cache_path.exists():
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except (json.JSONDecodeError, OSError):
            pass

    try:
        result = _llm_call_openai(body, model)
    except Exception as exc:
        # Do not cache transport / config errors -- they are not real LLM
        # judgments and would poison subsequent runs until cleared.
        return {"error": str(exc), "needs_human_review": True}

    if "error" not in result:
        try:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
        except OSError:
            pass
    return result


def enhance_with_llm(
    notices: list[dict[str, Any]],
    threshold: float = LLM_CONFIDENCE_THRESHOLD,
    model: str = LLM_MODEL,
) -> list[dict[str, Any]]:
    """For every notice with composite < threshold, run LLM validation and
    blend the result into composite_enhanced. Marks needs_human_review when
    any LLM sub-score is below 0.5.
    """
    for n in notices:
        scores = n.get("confidence_scores") or {}
        composite = scores.get("composite")
        if composite is None or composite >= threshold:
            continue
        res = llm_validate_notice(n, model=model)
        n["llm_validation"] = res
        sub = [
            res.get("coherence_score"),
            res.get("completeness_score"),
            res.get("single_notice_score"),
            res.get("legal_structure_score"),
        ]
        sub_num = [float(x) for x in sub if isinstance(x, (int, float))]
        if sub_num:
            llm_avg = sum(sub_num) / len(sub_num)
            scores["llm_semantic"] = round(llm_avg, 3)
            scores["composite_enhanced"] = round(
                composite * 0.6 + llm_avg * 0.4, 3
            )
            n["confidence_scores"] = scores
            if any(s < 0.5 for s in sub_num) or res.get("needs_human_review"):
                res["needs_human_review"] = True
    return notices


if ENABLE_LLM_VALIDATION and os.environ.get("OPENAI_API_KEY"):
    print("LLM validation enabled. Call enhance_with_llm(notices) after scoring.")
else:
    print("LLM validation defined but not active. Set ENABLE_LLM_VALIDATION=True "
          "and export OPENAI_API_KEY to use it.")

LLM validation enabled. Call enhance_with_llm(notices) after scoring.


### Run LLM validation on a single gazette

Load one already-parsed gazette, run `enhance_with_llm` only on its low-confidence notices (composite < 0.70), mirror `needs_human_review` to the notice top level so it is easy to query, and write the enhanced JSON back in place. Responses are cached under `.llm_cache/` so reruns cost nothing.

Change `target` to point at any folder in `output/`.

In [ ]:
target = "Kenya Gazette Vol CXXIVNo 282"
gpath = OUTPUT_DIR / target / f"{target}_gazette_spatial.json"

with open(gpath, "r", encoding="utf-8") as f:
    record = json.load(f)

notices = record["gazette_notices"]

below = [
    n for n in notices
    if (n.get("confidence_scores") or {}).get("composite", 1.0) < LLM_CONFIDENCE_THRESHOLD
]
print(f"{len(below)} of {len(notices)} notices below {LLM_CONFIDENCE_THRESHOLD:.2f} -- those will be sent to the LLM")

enhance_with_llm(notices)

# Mirror nested flag to the top level for easy filtering / DB ingest.
for n in notices:
    if (n.get("llm_validation") or {}).get("needs_human_review"):
        n["needs_human_review"] = True

flagged = [n for n in notices if n.get("needs_human_review")]
print(f"{len(flagged)} notices flagged for human review")

with open(gpath, "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)

print(f"Wrote enhanced JSON back to {gpath.name}")

# Show the flagged notices so you can spot-check them.
for n in flagged[:10]:
    scores = n.get("confidence_scores") or {}
    issues = (n.get("llm_validation") or {}).get("issues") or []
    print(
        f"  - notice {n.get('gazette_notice_no')}: "
        f"composite={scores.get('composite')} "
        f"enhanced={scores.get('composite_enhanced')} "
        f"issues={issues[:3]}"
    )

1 of 1 notices below 0.80 -- those will be sent to the LLM
1 notices flagged for human review
Wrote enhanced JSON back to Kenya Gazette Vol CIINo 83 - pre 2010_gazette_spatial.json
  - notice None: composite=0.253 enhanced=0.282 issues=['Text appears scrambled and incoherent.', 'Notice is truncated and does not end properly.', 'Multiple notices seem to be merged together.']


## Calibration and regression

Two small tools to validate the confidence system:

1. **Calibration** -- `sample_for_calibration()` draws a stratified sample of ~80-100 notices across the four bands and writes a YAML stub at `tests/calibration_sample.yaml`. Fill the `is_correct` field by hand for each notice, then `score_calibration()` computes precision/recall per band and prints suggested weight tweaks.

2. **Regression** -- `update_regression_fixture()` writes a snapshot of mean composite per canonical PDF to `tests/expected_confidence.json`. `check_regression()` reloads it and prints a warning when mean composite has dropped beyond the tolerance (default 0.05) for any fixture PDF.

In [20]:
import random


TESTS_DIR = PROJECT_ROOT / "tests"
CALIBRATION_FILE = TESTS_DIR / "calibration_sample.yaml"
REGRESSION_FILE = TESTS_DIR / "expected_confidence.json"

CANONICAL_PDFS = [
    "Kenya Gazette Vol CXINo 100",
    "Kenya Gazette Vol CXINo 103",
    "Kenya Gazette Vol CXIINo 76",
    "Kenya Gazette Vol CXXVIINo 63",
    "Kenya Gazette Vol CXXIVNo 282",
    "Kenya Gazette Vol CIINo 83 - pre 2010",
]


def sample_for_calibration(
    n_per_band: int = 20,
    seed: int = 42,
    output_root: Path = OUTPUT_DIR,
    out_path: Path = CALIBRATION_FILE,
) -> Path:
    """Draw a stratified sample across confidence bands and emit a YAML stub
    the user can hand-label with `is_correct: true/false` per notice.
    """
    random.seed(seed)
    bands: dict[str, list[dict[str, Any]]] = {"high": [], "medium": [], "low": []}
    for stem, _path, record in _iter_output_gazette_jsons(output_root):
        for n in record.get("gazette_notices") or []:
            c = (n.get("confidence_scores") or {}).get("composite")
            if c is None:
                continue
            band = "high" if c >= 0.80 else "medium" if c >= 0.50 else "low"
            bands[band].append({"pdf": stem, "notice": n})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    lines: list[str] = [
        "# Calibration sample for confidence scoring.",
        "# For each entry, set is_correct to true or false after manual review.",
        "# Then run score_calibration() to compute precision/recall per band.",
        "",
    ]
    total = 0
    for band in ("high", "medium", "low"):
        pool = bands[band]
        pick = random.sample(pool, min(n_per_band, len(pool)))
        for item in pick:
            n = item["notice"]
            scores = n.get("confidence_scores") or {}
            header = (n.get("gazette_notice_header") or "").replace('"', "'")[:80]
            lines.append(f"- pdf: \"{item['pdf']}\"")
            lines.append(f"  notice_no: \"{n.get('gazette_notice_no')}\"")
            lines.append(f"  header: \"{header}\"")
            lines.append(f"  composite: {scores.get('composite')}")
            lines.append(f"  band: {band}")
            lines.append(f"  is_correct: null  # true | false")
            lines.append("")
            total += 1

    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Wrote {total} notices to {out_path}")
    print("Edit the file to set is_correct for each, then call score_calibration().")
    return out_path


def _parse_calibration_yaml(path: Path) -> list[dict[str, Any]]:
    """Minimal YAML-subset parser -- we only emit the shape sample_for_calibration
    writes, so a dependency on PyYAML is unnecessary.
    """
    text = path.read_text(encoding="utf-8")
    entries: list[dict[str, Any]] = []
    current: dict[str, Any] | None = None
    for raw_line in text.splitlines():
        line = raw_line.rstrip()
        if not line or line.lstrip().startswith("#"):
            continue
        if line.startswith("- "):
            if current:
                entries.append(current)
            current = {}
            rest = line[2:]
            if ":" in rest:
                k, _, v = rest.partition(":")
                current[k.strip()] = _coerce(v.strip())
        elif current is not None and ":" in line:
            k, _, v = line.partition(":")
            current[k.strip()] = _coerce(v.strip())
    if current:
        entries.append(current)
    return entries


def _coerce(v: str) -> Any:
    if v.startswith('"') and v.endswith('"'):
        return v[1:-1]
    if v == "null":
        return None
    if v == "true":
        return True
    if v == "false":
        return False
    try:
        return float(v) if "." in v else int(v)
    except ValueError:
        return v


def score_calibration(path: Path = CALIBRATION_FILE) -> dict[str, Any]:
    """Compute precision/recall per band from a hand-labeled calibration file."""
    if not path.exists():
        print(f"{path} not found. Run sample_for_calibration() first.")
        return {}
    entries = _parse_calibration_yaml(path)
    labeled = [e for e in entries if isinstance(e.get("is_correct"), bool)]
    if not labeled:
        print("No entries have is_correct set. Fill in the file first.")
        return {}

    summary: dict[str, dict[str, int]] = {}
    for e in labeled:
        band = e.get("band", "?")
        summary.setdefault(band, {"n": 0, "correct": 0, "wrong": 0})
        summary[band]["n"] += 1
        if e["is_correct"]:
            summary[band]["correct"] += 1
        else:
            summary[band]["wrong"] += 1

    print(f"Labeled {len(labeled)}/{len(entries)} entries.")
    print(f"{'band':<8} {'n':>4} {'correct':>8} {'wrong':>6} {'precision':>10}")
    for band in ("high", "medium", "low"):
        s = summary.get(band)
        if not s:
            continue
        prec = s["correct"] / s["n"] if s["n"] else 0.0
        print(f"{band:<8} {s['n']:>4} {s['correct']:>8} {s['wrong']:>6} {prec:>10.2%}")

    hi = summary.get("high", {"n": 0, "correct": 0})
    if hi["n"] and hi["correct"] / hi["n"] < 0.85:
        print("\nHigh band precision below 85% -- consider tightening scorers:")
        print("  - raise structure deductions, or")
        print("  - lower the composite weight on notice_number.")
    low = summary.get("low", {"n": 0, "correct": 0})
    if low["n"] and low["correct"] / low["n"] > 0.3:
        print("\nLow band has many correct notices -- scorers are too strict.")
        print("  Consider relaxing structure or spatial deductions.")

    return summary


def update_regression_fixture(
    canonical: list[str] = CANONICAL_PDFS,
    out_path: Path = REGRESSION_FILE,
    output_root: Path = OUTPUT_DIR,
) -> Path:
    """Snapshot mean composite per canonical PDF for regression checks."""
    snapshot: dict[str, Any] = {}
    for name in canonical:
        path = output_root / name / f"{name}_gazette_spatial.json"
        if not path.exists():
            snapshot[name] = {"present": False}
            continue
        with open(path, "r", encoding="utf-8") as f:
            rec = json.load(f)
        agg = aggregate_confidence(rec.get("notices") or rec.get("gazette_notices") or [])
        doc_conf = rec.get("document_confidence") or {}
        snapshot[name] = {
            "present": True,
            "mean_composite": agg["mean"],
            "min_composite": agg["min"],
            "layout": doc_conf.get("layout"),
            "ocr_quality": doc_conf.get("ocr_quality"),
            "n_notices": agg["n"],
        }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    print(f"Wrote regression fixture: {out_path}")
    return out_path


def check_regression(
    tolerance: float = 0.05,
    fixture_path: Path = REGRESSION_FILE,
    output_root: Path = OUTPUT_DIR,
) -> bool:
    """Compare current mean composite per canonical PDF against the fixture.
    Returns True if all within tolerance; prints warnings otherwise.
    """
    if not fixture_path.exists():
        print(f"No fixture at {fixture_path}. Call update_regression_fixture() first.")
        return True
    expected = json.loads(fixture_path.read_text(encoding="utf-8"))
    regressed = False
    for name, snap in expected.items():
        if not snap.get("present"):
            continue
        cur_path = output_root / name / f"{name}_gazette_spatial.json"
        if not cur_path.exists():
            print(f"  missing current output for {name}")
            continue
        with open(cur_path, "r", encoding="utf-8") as f:
            rec = json.load(f)
        cur = aggregate_confidence(rec.get("notices") or rec.get("gazette_notices") or [])
        delta = cur["mean"] - float(snap["mean_composite"])
        if delta < -tolerance:
            regressed = True
            print(
                f"REGRESSION: {name} mean composite {snap['mean_composite']:.3f} -> {cur['mean']:.3f} "
                f"(drop {-delta:.3f})"
            )
        else:
            print(f"OK: {name} mean composite {cur['mean']:.3f} (baseline {snap['mean_composite']:.3f})")
    return not regressed


# Uncomment to generate a calibration sample or regression fixture.
# `sample_for_calibration()` overwrites tests/calibration_sample.yaml from
# scratch -- keep it commented so re-running the notebook does not erase
# any is_correct labels you have added by hand.
# sample_for_calibration()
update_regression_fixture()
# check_regression()

Wrote regression fixture: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\tests\expected_confidence.json


WindowsPath('C:/Users/Ronald Wahome/Documents/docling_spatial_pdfs/tests/expected_confidence.json')

In [21]:
check_regression()

OK: Kenya Gazette Vol CXINo 100 mean composite 0.990 (baseline 0.990)
OK: Kenya Gazette Vol CXINo 103 mean composite 0.989 (baseline 0.989)
OK: Kenya Gazette Vol CXIINo 76 mean composite 0.963 (baseline 0.963)
OK: Kenya Gazette Vol CXXVIINo 63 mean composite 0.977 (baseline 0.977)
OK: Kenya Gazette Vol CXXIVNo 282 mean composite 0.968 (baseline 0.968)
OK: Kenya Gazette Vol CIINo 83 - pre 2010 mean composite 0.253 (baseline 0.253)


True

In [ ]:
# F12 Test: Trailing Content Detector

print("=" * 70)
print("F12 TRAILING CONTENT DETECTOR - TEST SUITE")
print("=" * 70)

# T1: Happy path - pricing table at end
print("\n[T1] Testing Kenya Gazette Vol CXXIVNo 282 (trailing content expected)")
t1_path = OUTPUT_DIR / "Kenya Gazette Vol CXXIVNo 282" / "Kenya Gazette Vol CXXIVNo 282_spatial.txt"
if t1_path.exists():
    t1_text = t1_path.read_text(encoding="utf-8")
    t1_notices = split_gazette_notices(t1_text)
    
    last_notice = t1_notices[-1] if t1_notices else None
    if last_notice:
        full_text = last_notice.get("gazette_notice_full_text", "")
        has_subscription = "SUBSCRIPTION AND ADVERTISEMENT CHARGES" in full_text.upper()
        has_price = "PRICE: KSH" in full_text.upper()
        
        print(f"  Total notices: {len(t1_notices)}")
        print(f"  Last notice number: {last_notice.get('gazette_notice_no')}")
        print(f"  Last notice length: {len(full_text)} chars")
        print(f"  Contains 'SUBSCRIPTION AND ADVERTISEMENT CHARGES': {has_subscription}")
        print(f"  Contains 'PRICE: KSH': {has_price}")
        
        if not has_subscription and not has_price:
            print("  ✅ T1 PASS: Trailing content successfully excluded")
        else:
            print("  ❌ T1 FAIL: Trailing content still present in last notice")
    else:
        print("  ❌ T1 FAIL: No notices found")
else:
    print(f"  ⚠️  T1 SKIP: Test file not found at {t1_path}")

# T2: Edge case - no trailing content
print("\n[T2] Testing Kenya Gazette Vol CXINo 100 (no trailing content expected)")
t2_path = OUTPUT_DIR / "Kenya Gazette Vol CXINo 100" / "Kenya Gazette Vol CXINo 100_spatial.txt"
if t2_path.exists():
    t2_text = t2_path.read_text(encoding="utf-8")
    t2_notices = split_gazette_notices(t2_text)
    
    last_notice = t2_notices[-1] if t2_notices else None
    if last_notice:
        print(f"  Total notices: {len(t2_notices)}")
        print(f"  Last notice number: {last_notice.get('gazette_notice_no')}")
        print(f"  ✅ T2 PASS: No trailing content, notices unchanged")
    else:
        print("  ❌ T2 FAIL: No notices found")
else:
    print(f"  ⚠️  T2 SKIP: Test file not found at {t2_path}")

# T3: Degraded - function should not raise
print("\n[T3] Testing degraded input (should handle gracefully)")
try:
    degraded_text = "GAZETTE NOTICE NO. 1\nSome content\n" + ("X " * 100)
    degraded_notices = split_gazette_notices(degraded_text)
    print(f"  Total notices: {len(degraded_notices)}")
    print(f"  ✅ T3 PASS: Degraded input handled gracefully (no exception)")
except Exception as e:
    print(f"  ❌ T3 FAIL: Exception raised: {e}")

# T4: Multiple trailing sections (same as T1, but verify all excluded)
print("\n[T4] Testing multiple trailing sections (all should be excluded)")
if t1_path.exists():
    patterns = [
        "SUBSCRIPTION AND ADVERTISEMENT CHARGES",
        "PRICE: KSH",
        "IMPORTANT NOTICE TO SUBSCRIBERS",
        "GOVERNMENT PRINTER"
    ]
    
    last_notice = t1_notices[-1] if t1_notices else None
    if last_notice:
        full_text = last_notice.get("gazette_notice_full_text", "").upper()
        found_patterns = [p for p in patterns if p in full_text]
        
        if not found_patterns:
            print(f"  ✅ T4 PASS: All trailing patterns excluded")
        else:
            print(f"  ❌ T4 FAIL: Found trailing patterns: {found_patterns}")
    else:
        print("  ❌ T4 FAIL: No notices found")
else:
    print(f"  ⚠️  T4 SKIP: Test file not found")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

In [ ]:
# F14 Test Suite
import re
import json
from datetime import datetime, timezone
import time

print("=" * 80)
print("F14 Test Suite: Envelope Versioning Fields")
print("=" * 80)

# Test T1 & T2: Process CXXIVNo 282 and check presence + format
print("\n[T1 & T2] Processing Kenya Gazette Vol CXXIVNo 282...")
pdf_path = Path("Kenya Gazette PDFs/Kenya Gazette Vol CXXIVNo 282.pdf")
pipeline = GazettePipeline()
record1 = pipeline.process_pdf(pdf_path)

# Save output
output_dir = Path("output/Kenya Gazette Vol CXXIVNo 282")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "Kenya Gazette Vol CXXIVNo 282_gazette_spatial.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(record1, f, indent=2, ensure_ascii=False)

# T1: Check presence
required_keys = ["library_version", "schema_version", "extracted_at"]
missing = [k for k in required_keys if k not in record1]
if missing:
    print(f"  FAIL T1: Missing keys: {missing}")
else:
    print(f"  PASS T1: All three fields present")
    print(f"     library_version: {record1['library_version']}")
    print(f"     schema_version: {record1['schema_version']}")
    print(f"     extracted_at: {record1['extracted_at']}")

# T2: Format validation
lib_ver = record1["library_version"]
schema_ver = record1["schema_version"]
extracted = record1["extracted_at"]

t2_pass = True
if not re.fullmatch(r"\d+\.\d+\.\d+", lib_ver):
    print(f"  FAIL T2: library_version format invalid: {lib_ver}")
    t2_pass = False
elif not re.fullmatch(r"\d+\.\d+", schema_ver):
    print(f"  FAIL T2: schema_version format invalid: {schema_ver}")
    t2_pass = False
elif not re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z", extracted):
    print(f"  FAIL T2: extracted_at format invalid: {extracted}")
    t2_pass = False
else:
    try:
        dt = datetime.fromisoformat(extracted.replace("Z", "+00:00"))
        if dt.tzinfo != timezone.utc:
            print(f"  FAIL T2: extracted_at not UTC: {dt.tzinfo}")
            t2_pass = False
        else:
            print(f"  PASS T2: All format validations passed")
    except Exception as e:
        print(f"  FAIL T2: extracted_at not parseable: {e}")
        t2_pass = False

# Check F13 fields still present
if "pdf_sha256" in record1 and "gazette_issue_id" in record1:
    notices = record1.get("gazette_notices", [])
    if notices and all("notice_id" in n for n in notices):
        print(f"  PASS T1: F13 identity fields still intact")
    else:
        print(f"  FAIL T1: notice_id missing from some notices")
else:
    print(f"  FAIL T1: F13 fields missing")

# Test T3: Process same PDF twice - check determinism
print("\n[T3] Processing same PDF twice to test determinism...")
time.sleep(2)  # Ensure different timestamps
record2 = pipeline.process_pdf(pdf_path)

t3_pass = True
# Versions should be stable
if record1["library_version"] != record2["library_version"]:
    print(f"  FAIL T3: library_version differs: {record1['library_version']} vs {record2['library_version']}")
    t3_pass = False
if record1["schema_version"] != record2["schema_version"]:
    print(f"  FAIL T3: schema_version differs: {record1['schema_version']} vs {record2['schema_version']}")
    t3_pass = False

# extracted_at should differ
if record1["extracted_at"] == record2["extracted_at"]:
    print(f"  FAIL T3: extracted_at is identical (should differ): {record1['extracted_at']}")
    t3_pass = False

# F13 fields should match
if record1["pdf_sha256"] != record2["pdf_sha256"]:
    print(f"  FAIL T3: pdf_sha256 differs")
    t3_pass = False
if record1["gazette_issue_id"] != record2["gazette_issue_id"]:
    print(f"  FAIL T3: gazette_issue_id differs")
    t3_pass = False

notice_ids_1 = [n["notice_id"] for n in record1.get("gazette_notices", [])]
notice_ids_2 = [n["notice_id"] for n in record2.get("gazette_notices", [])]
if notice_ids_1 != notice_ids_2:
    print(f"  FAIL T3: notice_ids differ")
    t3_pass = False

if t3_pass:
    print(f"  PASS T3: versions stable, extracted_at differs, F13 ids identical")
    print(f"     Run 1 extracted_at: {record1['extracted_at']}")
    print(f"     Run 2 extracted_at: {record2['extracted_at']}")

# Test T4: Process other PDFs - cross-gazette consistency
print("\n[T4] Processing other PDFs for cross-gazette consistency...")
pdf_paths = [
    "Kenya Gazette PDFs/Kenya Gazette Vol CXINo 100.pdf",
    "Kenya Gazette PDFs/Kenya Gazette Vol CXIXNo 194.pdf",
]

records = [record1]  # Already have CXXIVNo 282
t4_pass = True

for pdf_path_str in pdf_paths:
    pdf_p = Path(pdf_path_str)
    if not pdf_p.exists():
        print(f"  WARNING: Skipping {pdf_p.name} (not found)")
        continue
    
    print(f"  Processing {pdf_p.name}...")
    r = pipeline.process_pdf(pdf_p)
    records.append(r)
    
    # Save output
    gazette_name = pdf_p.stem
    out_dir = Path(f"output/{gazette_name}")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{gazette_name}_gazette_spatial.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(r, f, indent=2, ensure_ascii=False)

# Check all have same library_version and schema_version
lib_versions = [r["library_version"] for r in records]
schema_versions = [r["schema_version"] for r in records]

if len(set(lib_versions)) != 1:
    print(f"  FAIL T4: library_version differs across PDFs: {lib_versions}")
    t4_pass = False
elif len(set(schema_versions)) != 1:
    print(f"  FAIL T4: schema_version differs across PDFs: {schema_versions}")
    t4_pass = False
elif lib_versions[0] != LIBRARY_VERSION or schema_versions[0] != SCHEMA_VERSION:
    print(f"  FAIL T4: unexpected version values: {lib_versions[0]}, {schema_versions[0]}")
    t4_pass = False
else:
    print(f"  PASS T4: All {len(records)} gazettes share library_version={LIBRARY_VERSION!r} and schema_version={SCHEMA_VERSION!r}")

# Final summary
print("\n" + "=" * 80)
all_pass = t2_pass and t3_pass and t4_pass
if all_pass:
    print("ALL F14 TESTS PASSED")
else:
    print("SOME F14 TESTS FAILED")
print("=" * 80)